# Modelo de Optimizacion para modem 4G

Instalacion paquete JuMP

In [ ]:
import Pkg # Importa el administrador de paquetes Pkg
Pkg.add("JuMP") # Instala el paquete JuMP

Instalacion de Solver

In [ ]:
Pkg.add("GLPK") # Instala el paquete GLPK

In [ ]:
using JuMP
using GLPK

# Crear el modelo de optimización usando el solver GLPK
modelo = Model(GLPK.Optimizer)

# Desactivar la salida de logs en la terminal
set_silent(modelo)

# Número total de máquinas (nodos)
n_nodos = 21

# Redundancia mínima de módems por nodo
redundancia_minima = 2

# Variables binarias: x[i] es 1 si se instala un módem, 0 en caso contrario
@variable(modelo, x[1:n_nodos], Bin)

# Definir la estructura del grafo como un arreglo de conjuntos
Grafo = [Set([i]) for i in 1:n_nodos]

# Función para agregar aristas al grafo
function add_edge!(Grafo, u, v)
    push!(Grafo[u], v) # Agrega v al conjunto de vecinos de u
    push!(Grafo[v], u) # Agrega u al conjunto de vecinos de v
end

# Grafo superior (Nodos 2 al 7)
edges_top = [
    (2,3), (2,4), (2,6), (2,7),
    (3,4), (3,5), (3,6),
    (4,5), (4,6),
    (5,6),
    (6,7)
]
for (u, v) in edges_top
    add_edge!(Grafo, u, v) # Agrega edges_top al grafo
end

# Grafo inferior (Nodos 9 al 21)
edges_down = [
    (9,10), (9,12),
    (10,11), (10,12),
    (11,12),
    (12,13), (12,14), (12,15), (12,16),
    (13,14), (13,15), (13,16), (13,17),(13,18), (13,19), (13,20),
    (14,15), (14,16), (14,17), (14,18), (14,19), (14,20), (14,21),
    (15,16), (15,17), (15,18), (15,19), (15,20), (15,21),
    (16,17), (16,18), (16,19), (16,20), (16,21),
    (17,18), (17,19), (17,20), (17,21),
    (18,19), (18,20), (18,21),
    (19,20), (19,21),
    (20,21)
]
for (u, v) in edges_down
    add_edge!(Grafo, u, v) # Agrega edges_down al grafo
end

# Función Objetivo: Minimizar el número total de módems en n máquinas (21)
@objective(modelo, Min, sum(x[i] for i in 1:n_nodos))

# Restricciones del Modelo (2 Casos):

# A) Nodos Aislados (1 y 8): Mínimo 1 módem
@constraint(modelo, sum(x[j] for j in Grafo[1]) >= 1) # X_1 >= 1
@constraint(modelo, sum(x[j] for j in Grafo[8]) >= 1) # X_8 >= 1

# B) Nodos Conectados (del 1 al 21 excepto el nodo 1 y 8): Redundancia mínima de 2 modems
for i in 1:n_nodos # Recorremos todos los nodos del grafo
    if i != 1 && i != 8 # Excluimos los nodos aislados
        @constraint(modelo, sum(x[j] for j in Grafo[i]) >= redundancia_minima) # X_i + X_j + ... + X_k >= 2
    end
end

# Resolver el problema
optimize!(modelo)
 
# Reporte de Resultados
println("==========================================")
println(" Estado de la Solución: ", termination_status(modelo))
println(" Redundancia mínima de Módems por maquina: ", redundancia_minima)
println(" Total de Módems requeridos como mínimo: ", Int(objective_value(modelo)))
println("==========================================")
println("Instalar módems en las siguientes máquinas:")

for i in 1:n_nodos
    if value(x[i]) > 0.5 # Comprobamos si la variable binaria es 1 (instalar módem)
        println("  -> Máquina x[$i]")
    end
end